In [1]:
import random

import numpy as np
import tensorflow as tf

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix


In [2]:
#Powtarzalność wyników
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
#wczytywanie zbioru iris
iris = load_iris()

X = iris.data #4 cechy wejściowe
y = iris.target #klasy: 0,1,2

class_names = iris.target_names
feature_names = iris.feature_names

print("Cechy: ")
for index, feature in enumerate(feature_names):
  print(f"{index+1}: {feature}")

print(f"\nRozmiar zbioru: {X.shape}")
print(f"Liczba klas: {len(class_names)}")

Cechy: 
1: sepal length (cm)
2: sepal width (cm)
3: petal length (cm)
4: petal width (cm)

Rozmiar zbioru: (150, 4)
Liczba klas: 3


In [5]:
#podział na zbiór treningowy i testowy
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = SEED,
    stratify = y
)

In [6]:
#standaryzacja danych
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
#budowa modelu sieci neuronowej klasyfikatora - Tensorflow
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,), name='flower_measurements'),
    tf.keras.layers.Dense(16, activation='relu', name='hidden_layer_1'),
    tf.keras.layers.Dense(8, activation='relu', name='hidden_layer_2'),
    tf.keras.layers.Dense(3, activation='softmax', name='output_layer')
])

In [8]:
#kompilacja modelu
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = [
        "accuracy"
    ]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer_1 (Dense)          │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
#mechanizm early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience = 20,
    restore_best_weights = True
)


In [10]:
#trening modelu
history  = model.fit(
    X_train_scaled,
    y_train,
    validation_split = 0.2,
    epochs = 300,
    batch_size = 8,
    callbacks = [early_stopping],
    verbose = 1
)


Epoch 1/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.1562 - loss: 1.2998 - val_accuracy: 0.1667 - val_loss: 1.2377
Epoch 2/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1979 - loss: 1.1854 - val_accuracy: 0.2500 - val_loss: 1.1359
Epoch 3/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3021 - loss: 1.0910 - val_accuracy: 0.3333 - val_loss: 1.0559
Epoch 4/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3229 - loss: 1.0144 - val_accuracy: 0.4167 - val_loss: 0.9948
Epoch 5/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3750 - loss: 0.9529 - val_accuracy: 0.3750 - val_loss: 0.9430
Epoch 6/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4896 - loss: 0.8977 - val_accuracy: 0.5417 - val_loss: 0.8952
Epoch 7/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6042 - loss: 0.8453 - val_accuracy: 0.5833 - val_loss: 0.8499
Epoch 8/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6771 - loss: 0.7946 - val_accuracy: 0.5833 - 